In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Load the dataset
#df = pd.read_csv('your_dataset.csv')
df = pd.read_excel('Wind Data - Dec - Jan.xlsx')

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder

# Load the dataset
# file_path = 'Wind Data - Dec - Jan.xlsx'
# df = pd.read_excel(file_path)

# Encode the 'Wtg' column as it is a string
label_encoder = LabelEncoder()
df['Wtg_encoded'] = label_encoder.fit_transform(df['Wtg'])

# Perform clustering based on the encoded 'Wtg' (Turbine ID)
kmeans = KMeans(n_clusters=10, random_state=0).fit(df[['Wtg_encoded']])

# Add the cluster labels to the dataframe
df['Cluster'] = kmeans.labels_

# Create a dictionary to map cluster labels to 'Wtg' names
cluster_to_wtg = {i: df[df['Cluster'] == i]['Wtg'].mode()[0] for i in range(10)}

# Map the cluster labels to 'Wtg' names
df['Cluster'] = df['Cluster'].map(cluster_to_wtg)

# Drop the encoded column as it's no longer needed
df.drop(columns=['Wtg_encoded'], inplace=True)

# Display all columns including 'Timestamp'
print(df)

# Save the clustered data to a new Excel file
#df.to_excel('Clustered_Wind_Data.xlsx', index=False)


In [ ]:
# Function to plot P_ACT_Avg vs V_WIN_Avg for a given date range with negative values and continuous zeros in different colors
def plot_power_curve(start_date, end_date):
    # Convert the 'Timestamp' column to datetime
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    
    # Filter the data for the given date range
    mask = (df['Timestamp'] >= start_date) & (df['Timestamp'] <= end_date)
    filtered_df = df[mask]
    
    # Plot P_ACT_Avg vs V_WIN_Avg for each cluster in the given date range
    for cluster in filtered_df['Cluster'].unique():
        cluster_data = filtered_df[filtered_df['Cluster'] == cluster]
        plt.figure(figsize=(10, 6))
        plt.scatter(cluster_data[cluster_data['P_ACT_Avg'] > 0]['V_WIN_Avg'], 
                    cluster_data[cluster_data['P_ACT_Avg'] > 0]['P_ACT_Avg'], 
                    label=f'Cluster {cluster}', color='blue')
        plt.scatter(cluster_data[cluster_data['P_ACT_Avg'] < 0]['V_WIN_Avg'], 
                    cluster_data[cluster_data['P_ACT_Avg'] < 0]['P_ACT_Avg'], 
                    label=f'Negative Values Cluster {cluster}', color='red')
        
        # Identify continuous zeros and plot them in a different color
        zero_mask = (cluster_data['P_ACT_Avg'] == 0)
        continuous_zeros = zero_mask & zero_mask.shift(-1, fill_value=False)
        plt.scatter(cluster_data[continuous_zeros]['V_WIN_Avg'], 
                    cluster_data[continuous_zeros]['P_ACT_Avg'], 
                    label=f'Continuous Zeros Cluster {cluster}', color='green')
        
        plt.xlabel('Nacelle Wind Speed (V_WIN_Avg)')
        plt.ylabel('Actual Power (P_ACT_Avg)')
        plt.title(f'Power Curve for Cluster {cluster} from {start_date} to {end_date}')
        plt.legend()
        plt.show()

# Example usage: Plot power curve for a given date range
plot_power_curve('2022-11-26', '2023-01-02')